# Katyayani: which service caused the incident?

200 real fault injections on an 8-service Kubernetes cluster. For each incident, rank the services so the injected root comes first. Two rankers: the **loudest service** (largest metric shift, no learning) and a **learned ranker** tested leave-one-fault-out, so it never trains on the fault type it is scored on.

Katyayani is one of the nine models of **Sati**, the investigation agent of [Pashupatastra](https://github.com/preetraval45/PASHUPATASTRA), each named for a form of Navadurga. This notebook reproduces its core result on the public dataset, with the split that keeps the test honest.

In [ ]:
import glob, os
def find(pattern):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits, f"{pattern} not found under /kaggle/input"
    return hits[0]
import numpy as np, pandas as pd
incidents = pd.read_csv(find("incidents.csv"))
metrics = pd.read_parquet(find("metrics.parquet"))
incidents = incidents[incidents["error"].isna() | (incidents["error"] == "")]
print(len(incidents), "incidents;", metrics.shape, "metric rows")
incidents["fault"].value_counts()

## Features: how far each metric moved, against its own baseline

For each service and metric: the robust shift of the median after injection against the 2 minutes before, in units of the baseline's spread. Log-compressed, because steady baselines make raw z-scores enormous.

In [ ]:
METRICS = ["cpu_cores", "memory_bytes", "latency_p90_ms", "requests_per_s", "error_rate"]
rows = []
for inc in incidents.itertuples():
    m = metrics[metrics["incident"] == inc.incident]
    before, after = m[m["time"] < inc.inject_time], m[m["time"] >= inc.inject_time]
    for svc in m["service"].unique():
        b, a = before[before["service"] == svc], after[after["service"] == svc]
        row = {"incident": inc.incident, "fault": inc.fault, "service": svc, "root": int(svc == inc.root_service)}
        for c in METRICS:
            base = b[c].dropna(); post = a[c].dropna()
            if len(base) < 2 or not len(post):
                row[c] = 0.0; continue
            scale = (base.quantile(.75) - base.quantile(.25)) / 1.349 or base.std() or 1e-6
            z = (post.median() - base.median()) / (scale if scale > 0 else 1e-6)
            row[c] = float(np.sign(z) * np.log1p(abs(z)))
        rows.append(row)
table = pd.DataFrame(rows)
table["peak"] = table[METRICS].abs().max(axis=1)
table.head()

In [ ]:
def rca_at_1(scored, score):
    top = scored.loc[scored.groupby("incident")[score].idxmax()]
    return top["root"].mean()

print(f"loudest service RCA@1: {rca_at_1(table, 'peak'):.3f}   (chance: {1 / table.groupby('incident').size().mean():.3f})")

## A learned ranker, leave one fault type out

In [ ]:
import lightgbm as lgb
FEATS = METRICS + ["peak"]
table["rank_in_incident"] = table.groupby("incident")["peak"].rank(ascending=False)
FEATS.append("rank_in_incident")
results = {}
for fault in sorted(table["fault"].unique()):
    train, test = table[table["fault"] != fault].sort_values("incident"), table[table["fault"] == fault].copy()
    model = lgb.LGBMRanker(n_estimators=200, learning_rate=0.05, num_leaves=15, verbose=-1)
    model.fit(train[FEATS], train["root"], group=train.groupby("incident").size().values)  # sorted by incident
    test["score"] = model.predict(test[FEATS])
    results[fault] = {"learned": rca_at_1(test, "score"), "loudest": rca_at_1(test, "peak")}
pd.DataFrame(results).T.round(3)

**Reading it:** the loudest service is a strong rule on this stack, because a fault here mostly makes its own service move most. The bad-release fault is the hard case: a failing release raises errors at every caller up the chain. Katyayani, trained on *other* systems (RCAEval), ranks the root first in 0.895 of these incidents; see the project's paper for that sim-to-real result.